# World-indepth: Colab View Budget Challenge

**Open in Colab**: this notebook runs the first real MapAnything experiment on a user-selected GPU runtime. It does not upload private photographs or generated results to GitHub.

The experiment is deliberately staged: complete the two-view smoke test first, then manually run the separate full-budget cell for `1, 2, 3, 5, 8` views.


## A. Project and environment setup

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import platform
import subprocess
import sys
import time

REPO_URL = "https://github.com/HimanshuPathak2725/World-indepth.git"
REPO_BRANCH = "feat/view-budget-challenge"
REPO_DIR = Path("/content/World-indepth")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR, check=True)

REPO_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print({"repository": REPO_URL, "branch": REPO_BRANCH, "sha": REPO_SHA,
       "recorded_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
       "python": sys.version, "platform": platform.platform()})
assert REPO_SHA, "Could not record the checked-out repository SHA"
LAB_DIR = REPO_DIR / "spatial-reconstruction-lab"
sys.path.insert(0, str(LAB_DIR))
from dataset import select_input_views, validate_manifest, validate_budget_inputs
from mapanything_adapter import (MODEL_ID, MODEL_REVISION, OUTPUT_KEYS,
                                  prediction_summary, save_predictions,
                                  validate_complete_prediction)


## B. GPU preflight

This cell fails rather than silently falling back to CPU. Colab GPU types vary, so the detected device and memory are recorded for every experiment.


In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is connected. In Colab choose Runtime > Change runtime type > T4 GPU (or another GPU) and rerun.")

device = torch.device("cuda")
props = torch.cuda.get_device_properties(device)
free_vram, total_vram = torch.cuda.mem_get_info(device)
bf16_supported = bool(torch.cuda.is_bf16_supported())
ram_gb = None
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except ImportError:
    pass
disk = shutil.disk_usage("/content")
print({
    "gpu": props.name,
    "total_vram_gib": round(total_vram / 2**30, 2),
    "free_vram_gib": round(free_vram / 2**30, 2),
    "cuda_available": True,
    "cuda_version": torch.version.cuda,
    "bf16_supported": bf16_supported,
    "system_ram_gib": None if ram_gb is None else round(ram_gb, 2),
    "disk_free_gib": round(disk.free / 2**30, 2),
})


## C. Install the pinned MapAnything environment

The upstream repository was inspected at source commit `3d10cf7a3016fc0f9bb13a071ee66c47b10be0d9`. The Apache checkpoint is pinned independently to Hugging Face revision `00f9c245bbcb60522d1ed7f9e9d88462c6e3f38a`. The installation uses the upstream editable package without replacing the working Colab torch/CUDA stack.


In [ ]:
UPSTREAM_URL = "https://github.com/facebookresearch/map-anything.git"
UPSTREAM_COMMIT = "3d10cf7a3016fc0f9bb13a071ee66c47b10be0d9"
CHECKPOINT_ID = "facebook/map-anything-apache"
CHECKPOINT_REVISION = "00f9c245bbcb60522d1ed7f9e9d88462c6e3f38a"
UPSTREAM_DIR = Path("/content/map-anything")
if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", UPSTREAM_COMMIT], cwd=UPSTREAM_DIR, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_DIR, text=True).strip() == UPSTREAM_COMMIT

# --no-deps avoids replacing Colab's already validated CUDA/PyTorch installation.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(UPSTREAM_DIR), "--no-deps"], check=True)
import importlib
mapanything = importlib.import_module("mapanything")
from mapanything import MapAnything
from mapanything.utils.image import load_images
print({"mapanything_import": mapanything.__file__, "source_commit": UPSTREAM_COMMIT,
       "checkpoint": CHECKPOINT_ID, "checkpoint_revision": CHECKPOINT_REVISION,
       "torch": torch.__version__, "torchvision": __import__("torchvision").__version__})
assert CHECKPOINT_ID == MODEL_ID and CHECKPOINT_REVISION == MODEL_REVISION


## D. Upload and validate the private scene

Upload exactly one ZIP to this Colab runtime. It must contain eight candidate input photographs and four held-out photographs (or a manifest naming them). The helper below creates a manifest only when one is absent; scene origin and licence are explicit user-provided fields.


In [ ]:
from google.colab import files
import zipfile
from PIL import Image

uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".zip"):
    raise ValueError("Upload exactly one .zip file containing the private scene")
zip_name = next(iter(uploaded))
SCENE_DIR = Path("/content/private_scene")
SCENE_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(SCENE_DIR)

manifest_candidates = list(SCENE_DIR.rglob("manifest.json")) + list(SCENE_DIR.rglob("scene_manifest.json"))
if manifest_candidates:
    MANIFEST_PATH = manifest_candidates[0]
    SCENE_ROOT = MANIFEST_PATH.parent
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    origin = input("Scene origin (for example: personal phone capture): ").strip()
    licence = input("Scene/image licence (for example: owner permission): ").strip()
    if not origin or not licence:
        raise ValueError("Scene origin and licence are required to create a manifest")
    image_paths = sorted(p for p in SCENE_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
    if len(image_paths) != 12:
        raise ValueError(f"Expected exactly 12 photographs (8 input + 4 held-out), found {len(image_paths)}")
    views = []
    for index, path in enumerate(image_paths):
        with Image.open(path) as image:
            width, height = image.size
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        views.append({"id": f"view-{index:03d}", "filename": str(path.relative_to(SCENE_DIR)),
                      "capture_order": index, "sha256": digest, "width": width, "height": height,
                      "role": "input" if index < 8 else "held_out"})
    manifest = {"schema_version": 1, "scene_id": SCENE_DIR.name,
                "origin": {"description": origin, "license": licence},
                "views": views[:8], "held_out": views[8:]}
    MANIFEST_PATH = SCENE_DIR / "manifest.json"
    SCENE_ROOT = SCENE_DIR
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n")

validate_manifest(manifest, SCENE_ROOT)
if len(manifest["views"]) != 8 or len(manifest["held_out"]) != 4:
    raise ValueError("The experiment requires exactly eight inputs and four held-out photographs")
input_ids = {view["id"] for view in manifest["views"]}
held_out_ids = {view["id"] for view in manifest["held_out"]}
assert not input_ids & held_out_ids
print({"manifest": str(MANIFEST_PATH), "scene_id": manifest["scene_id"],
       "inputs": sorted(input_ids), "held_out": sorted(held_out_ids),
       "origin": manifest["origin"], "license": manifest["origin"]["license"]})


## E. Two-view smoke test (run this first)

The smoke test uses exactly two input images and stops on any incomplete or malformed model output. It does not start the full challenge after failure.


In [ ]:
def telemetry():
    return {
        "allocated_bytes": int(torch.cuda.memory_allocated()),
        "reserved_bytes": int(torch.cuda.memory_reserved()),
    }

def image_path(view):
    return SCENE_DIR / view["file"]

def run_inference(paths):
    amp_dtype = "bf16" if bf16_supported else "fp16"
    views = load_images([str(path) for path in paths])
    start = time.perf_counter()
    model = MapAnything.from_pretrained(CHECKPOINT_ID, revision=CHECKPOINT_REVISION).to(device)
    load_seconds = time.perf_counter() - start
    torch.cuda.reset_peak_memory_stats()
    before = telemetry()
    start = time.perf_counter()
    predictions = model.infer(views, memory_efficient_inference=True, minibatch_size=1,
                              use_amp=True, amp_dtype=amp_dtype)
    torch.cuda.synchronize()
    inference_seconds = time.perf_counter() - start
    after = telemetry()
    complete = []
    for prediction in predictions:
        validate_complete_prediction(prediction)
        missing = set(OUTPUT_KEYS) - prediction.keys()
        if missing:
            raise RuntimeError(f"Model output is incomplete: {sorted(missing)}")
        complete.append(prediction)
    return model, complete, {
        "model_load_seconds": load_seconds, "inference_seconds": inference_seconds,
        "amp_dtype": amp_dtype, "input_count": len(paths),
        "memory_before": before, "memory_after": after,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
        "peak_reserved_bytes": int(torch.cuda.max_memory_reserved()),
    }

smoke_views = select_input_views(manifest, 2)
smoke_paths = [image_path(view) for view in smoke_views]
assert len(smoke_paths) == 2
smoke_model, smoke_predictions, smoke_telemetry = run_inference(smoke_paths)
smoke_summary = prediction_summary(smoke_predictions)
smoke_summary.update(smoke_telemetry)
smoke_summary["input_ids"] = [view["id"] for view in smoke_views]
smoke_summary["input_sha256"] = [view["sha256"] for view in smoke_views]
smoke_summary["output_shapes"] = {key: list(next(iter(smoke_predictions))[key].shape) for key in OUTPUT_KEYS}
smoke_summary["coordinate_convention"] = "OpenCV RDF camera-to-world (+X right, +Y down, +Z forward)"
print(json.dumps(smoke_summary, indent=2, default=str))


## F. Full View Budget Challenge (manual cell)

Run this cell only after the smoke test above succeeds. It runs nested, deterministic, spatially spread subsets and never uses held-out views. A failed budget is recorded without deleting successful earlier runs.


In [ ]:
# Manual execution gate: change this only after smoke_summary exists and is complete.
if "smoke_summary" not in globals():
    raise RuntimeError("Run the two-view smoke test successfully before the full challenge")

RESULTS_DIR = Path("/content/world_indepth_results")
RESULTS_DIR.mkdir(exist_ok=True)

def persist_run_outputs(predictions, output_dir):
    import numpy as np
    output_dir.mkdir(parents=True, exist_ok=True)
    compact_points = []
    for prediction in predictions:
        points = prediction["pts3d"].detach().cpu().numpy()[0]
        mask = prediction["mask"].detach().cpu().numpy()[0]
        if mask.ndim == 3:
            mask = mask[..., 0]
        rgb_value = prediction.get("img_no_norm")
        if rgb_value is None:
            rgb = np.zeros((*points.shape[:2], 3), dtype=np.uint8)
        else:
            rgb = rgb_value.detach().cpu().numpy()[0]
            if rgb.max() <= 1.0:
                rgb = rgb * 255
            rgb = rgb.clip(0, 255).astype(np.uint8)
        compact_points.extend(
            (0, 0, tuple(map(float, xyz)), tuple(map(int, color)))
            for xyz, color, keep in zip(points.reshape(-1, 3), rgb.reshape(-1, 3), mask.reshape(-1))
            if keep
        )
    write_compact(str(output_dir / "pointcloud.wipcld"), compact_points)
    np.savez_compressed(
        output_dir / "camera_parameters.npz",
        intrinsics=np.concatenate([p["intrinsics"].detach().cpu().numpy() for p in predictions]),
        camera_poses=np.concatenate([p["camera_poses"].detach().cpu().numpy() for p in predictions]),
        cam_trans=np.concatenate([p["cam_trans"].detach().cpu().numpy() for p in predictions]),
        cam_quats=np.concatenate([p["cam_quats"].detach().cpu().numpy() for p in predictions]),
        conf=np.concatenate([p["conf"].detach().cpu().numpy() for p in predictions]),
        mask=np.concatenate([p["mask"].detach().cpu().numpy() for p in predictions]),
    )
run_records = []
for budget in (1, 2, 3, 5, 8):
    selected = select_input_views(manifest, budget)
    assert len(selected) == budget
    validate_budget_inputs(manifest, [v["id"] for v in selected], budget)
    assert not {v["id"] for v in selected} & {v["id"] for v in manifest["held_out"]}
    record = {"budget": budget, "input_ids": [v["id"] for v in selected],
              "input_sha256": [v["sha256"] for v in selected],
              "settings": {"memory_efficient_inference": True, "minibatch_size": 1, "use_amp": True}}
    try:
        _, predictions, telemetry_data = run_inference([image_path(v) for v in selected])
        output_dir = RESULTS_DIR / f"views_{budget}"
        record.update(telemetry_data)
        record.update(prediction_summary(predictions))
        record.update({
            "status": "success", "model_revision": CHECKPOINT_REVISION,
            "source_revision": UPSTREAM_COMMIT, "output_dir": str(output_dir),
            "persisted_arrays": save_predictions(predictions, output_dir),
        })
        persist_run_outputs(predictions, output_dir)
        (output_dir / "run_manifest.json").write_text(json.dumps({
            "manifest": manifest, "configuration": record
        }, indent=2, default=str) + "\n")
    except Exception as error:
        record.update({"status": "failed", "error_type": type(error).__name__, "error": str(error)})
        print(f"Budget {budget} failed; successful previous runs were retained: {error}")
    run_records.append(record)
Path(RESULTS_DIR / "run_manifest.json").write_text(json.dumps(run_records, indent=2, default=str) + "\n")
print(json.dumps(run_records, indent=2, default=str))


## G. Result export

Only validated predictions are exported. The archive stays in the Colab runtime unless you explicitly download it.


In [ ]:
import shutil

if not RESULTS_DIR.exists():
    raise RuntimeError("No results directory exists; run the smoke test and/or full challenge first")
metadata = {"repository_sha": REPO_SHA, "source_revision": UPSTREAM_COMMIT,
            "model_id": CHECKPOINT_ID, "model_revision": CHECKPOINT_REVISION,
            "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            "scene_id": manifest["scene_id"], "held_out_ids": [v["id"] for v in manifest["held_out"]],
            "coordinate_convention": "OpenCV RDF camera-to-world",
            "runs": run_records if "run_records" in globals() else [],
            "smoke_test": smoke_summary if "smoke_summary" in globals() else None}
(RESULTS_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str) + "\n")
(RESULTS_DIR / "experiment_summary.json").write_text(json.dumps({
    "smoke_test": smoke_summary if "smoke_summary" in globals() else None,
    "full_runs": run_records if "run_records" in globals() else []
}, indent=2, default=str) + "\n")
archive = shutil.make_archive("/content/world_indepth_results", "zip", RESULTS_DIR)
print(archive)


In [ ]:
from google.colab import files
files.download("/content/world_indepth_results.zip")


## H. Final presentation

Values are intentionally not filled until the corresponding run completes. Smoke-test results and full-budget results must remain visibly separate. Image similarity metrics (PSNR/SSIM/LPIPS) are not reported here because this notebook does not register held-out camera poses and a validated renderer.


In [ ]:
rows = []
for budget in (1, 2, 3, 5, 8):
    completed = next((r for r in globals().get("run_records", [])
                      if r.get("budget") == budget and r.get("status") == "success"), None)
    rows.append({"Views": budget, "Status": "not run" if completed is None else "success",
                 "Runtime (s)": None if completed is None else completed.get("inference_seconds"),
                 "Peak VRAM (GiB)": None if completed is None else round(completed.get("peak_reserved_bytes", 0) / 2**30, 3),
                 "Valid points": None if completed is None else completed.get("valid_point_count")})
print("Smoke test (separate):", json.dumps(globals().get("smoke_summary", None), indent=2, default=str))
print("Full View Budget Challenge:")
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    print(json.dumps(rows, indent=2))
